In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing

* Add `.gitignore` to the repository.
* Add `README.md` and keep the repository private until the project is complete.
* Add `requirements.txt` and create the project virtual environment.

# Original Sales Dataset

In [ ]:
banco_inicial = pd.read_csv('data/raw/data.csv')

print(f"O banco de dados possui {banco_inicial.shape[0]} linhas e {banco_inicial.shape[1]} colunas.")

In [ ]:
banco_inicial.info()

* Several fields have incorrect data types. The numeric fields are `valor_unitario`, `quantidade`, `valor_total`, `preco_custo`, and `saldo`.
* Convert `data_venda` to datetime so it can be used in pandas time-series operations.

In [ ]:
banco_inicial.rename(columns={'produto': 'id_produto'}, inplace=True)

In [ ]:
banco_inicial['grupo'].value_counts()

In [ ]:
banco_inicial['grupo'].value_counts(normalize=True)*100

The original dataset contains a small number of records in irregular or poorly defined groups:

- SEM GRUPO: 5.71%
- PRODUTO ESPECIAL: 0.75%
- TV BOX: 0.81%
- REDES: 0.02%

These groups are excluded from the main forecasting analysis because the project focuses on consistent sales patterns and demand-related revenue. The remaining groups provide a more reliable basis for comparison and forecasting.

# Dataset Version 1

In [ ]:
banco = banco_inicial[
    ~banco_inicial['grupo'].isin(['SEM GRUPO', 'PRODUTO ESPECIAL', 'TV BOX', 'REDES'])
]

In [ ]:
banco['grupo'].value_counts(normalize=True)*100

The groups with the largest record counts are not necessarily the most relevant for revenue analysis. CABO, ANTENA, and COMPONENTES ANTENAS have many transactions, but their products are often sold by length or in low-value units. A smaller group such as SEGURANCA may generate comparable revenue with far fewer transactions.

The analysis therefore considers both sales volume and revenue contribution rather than using record count alone.

## Data Preprocessing

In [ ]:
# Convert monetary columns to numeric values

colunas_monetarias = [
    'valor_unitario',
    'valor_total',
    'preco_custo',
    'saldo'
]

for col in colunas_monetarias:
    banco[col] = (
        banco[col]
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    banco[col] = pd.to_numeric(
        banco[col],
        errors='coerce' # Convert non-numeric values to NaN
    )

In [ ]:
# Convert the date column to datetime
banco['data_venda'] = pd.to_datetime(
    banco['data_venda'],
    errors='coerce' # Convert invalid values to NaT
)

In [ ]:
banco.info()

---

After the initial type conversion, the dataset still contains missing values. The number of product IDs is also lower than the number of records, which indicates that a product may appear under more than one identifier.

The main fields requiring validation are `valor_unitario`, `quantidade`, `valor_total`, `preco_custo`, and `saldo`.

In [ ]:
banco[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

In [ ]:
banco.groupby('grupo')[['preco_custo', 'saldo']].apply(
    lambda x: x.isna().sum()
)

Many `preco_custo` and `saldo` values are missing, which prevents a reliable profit analysis. The project therefore focuses on revenue, seasonality, and demand forecasting instead of profit forecasting.

Because `valor_total = valor_unitario * quantidade`, some missing values can be reconstructed when the other two fields are available.

In [ ]:
mask_total = (
    banco['valor_total'].isna()
    & banco['valor_unitario'].notna()
    & banco['quantidade'].notna()
)

banco.loc[mask_total, 'valor_total'] = (
    banco.loc[mask_total, 'valor_unitario']
    * banco.loc[mask_total, 'quantidade']
)

mask_unitario = (
    banco['valor_unitario'].isna()
    & banco['valor_total'].notna()
    & banco['quantidade'].notna()
    & (banco['quantidade'] > 0)
)

banco.loc[mask_unitario, 'valor_unitario'] = (
    banco.loc[mask_unitario, 'valor_total']
    / banco.loc[mask_unitario, 'quantidade']
)

In [ ]:
banco[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

The imputation step reduced missing `valor_unitario` values from 1,821 to 753 (58.65% improvement) and missing `valor_total` values from 1,291 to 865 (33% improvement).

In [ ]:
# Remove records with missing value_total, value_unitario, and quantity
banco = banco.dropna(
    subset=['valor_unitario', 'quantidade', 'valor_total'],
    how='all'
)

In [ ]:
# number of records restantes in the dataset of date
banco.shape[0]

In [ ]:
# Records with value_total populated but value_unitario and quantity missing
mask = (
    banco['valor_total'].notna()
    & banco['valor_unitario'].isna()
    & banco['quantidade'].isna()
)

mask.sum()

Records with `valor_total` but without both `valor_unitario` and `quantidade` provide insufficient information for reliable analysis. Because they represent a small portion of the dataset, they are removed.

In [ ]:
# Remove records with value_total populated but value_unitario and quantity missing

banco = banco[~mask].copy()

In [ ]:
# Records with missing quantity but populated value_unitario and value_total

mask_qtdna = (
    banco['valor_unitario'].notna()
    & banco['quantidade'].isna()
    & banco['valor_total'].notna()
)

mask_qtdna.sum()

In [ ]:
qtd_calc = (
    banco.loc[mask_qtdna, 'valor_total']
    / banco.loc[mask_qtdna, 'valor_unitario']
)
print(qtd_calc)

The calculated quantities include fractional values, which are inconsistent with the way products are sold in this business. This suggests that discounts or adjustments may have been applied at the time of sale.

Creating an adjustment variable for only a small fraction of records would add noise to the forecasts. The records with missing `valor_unitario`, `quantidade`, or `valor_total` are therefore removed.

In [ ]:
banco = banco.dropna(
    subset=['valor_unitario', 'quantidade', 'valor_total']
).copy()

In [ ]:
banco.shape[0]

After removing missing values and irregular groups, approximately 87.31% of the original records remain. This is a sufficiently large and consistent base for the forecasting analysis.

In [ ]:
banco.info()

The remaining dataset requires additional validation:

1. Records without `data_venda` must be removed because the project uses time-series methods.
2. Missing `preco_custo` and `saldo` values prevent profit forecasting, so these fields are not used as machine-learning inputs.
3. Null or non-positive values in `valor_unitario`, `quantidade`, or `valor_total` are treated as inconsistencies and removed.

#### Null and/or Missing Sale Dates

In [ ]:
# number of records with sales records missing
mask_datna = (
    banco['data_venda'].isna()
)

mask_datna.sum()

In [ ]:
# Removing the records with dates missing
banco = banco[~mask_datna].copy()

#### Null or Negative Quantity

In [ ]:
# number of records with quantity lower or igual a zero
banco[banco['quantidade'] <= 0].shape[0]

#### Null or Negative Unit Price

In [ ]:
# number of records with values_unitarios lower or igual a zero
banco[banco['valor_unitario'] <= 0].shape[0]

In [ ]:
banco[banco['valor_unitario'] <= 0]

#### Null or Negative Total Value

In [ ]:
# number of records with value_total lower or igual a zero
banco[banco['valor_total'] <= 0].shape[0]

In [ ]:
banco[banco['valor_total'] <= 0]

#### Removing Inconsistencies

In [ ]:
banco.drop([3658, 5877], inplace=True)

In [ ]:
banco.info()

#### Identifying Outliers by Group and Product

In [ ]:
banco.groupby('grupo')['valor_unitario'].describe()

In [ ]:
banco.groupby('grupo')['quantidade'].describe()

The maximum values are far from the median because the dataset mixes unit-based and length-based sales. A single high-value product can generate revenue comparable to many units of another product.

In [ ]:
banco.loc[
    banco['valor_unitario'] > 1000,
    ['grupo','descricao','valor_unitario','quantidade','valor_total']
].sort_values('valor_unitario', ascending=False)

In [ ]:
banco.loc[
    banco['quantidade'] > 50,
    ['grupo','descricao','quantidade','valor_unitario','valor_total']
].sort_values('quantidade', ascending=False)

The highest-value items are mainly sold by unit and are concentrated in groups such as PORTAO. In contrast, groups with many transactions often contain products sold by length or in low-value units. These observations do not indicate invalid records; they reflect different sales mechanisms.

---

# Exploratory Data Analysis

Profit analysis is out of scope, so `preco_custo` and `saldo` are excluded from the predictive features.

In [ ]:
banco_pred = banco.drop(['preco_custo', 'saldo'], axis=1)

In [ ]:
banco_pred.head()

## Sales Analysis

The objective is to compare groups using complementary metrics: units sold, number of transactions, revenue, and average ticket. A high quantity does not necessarily mean high revenue because some products are sold by length, while a single transaction may generate substantial revenue.

### Quantity Sold by Group

In [ ]:
# quantity sold by group
banco_pred.groupby('grupo')['quantidade'].sum().sort_values(ascending=False)

In [ ]:
quandidade_vendida = banco_pred.groupby('grupo')['quantidade'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(quandidade_vendida.index, quandidade_vendida.values)
plt.title('quantity sold by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.ylabel('quantity sold')
plt.show()

CABO behaves like a structural outlier because many of its products are sold by length. Its high volume is legitimate and should be interpreted differently from unit-based product groups.

---

### Number of Sales by Group

In [ ]:
# number of sales by group
banco_pred.groupby('grupo').size().sort_values(ascending=False)

In [ ]:
numero_vendas = banco_pred.groupby('grupo').size().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(numero_vendas.index, numero_vendas.values)
plt.title('number of sales by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.ylabel('number of sales')
plt.show()

---

### Revenue by Group

In [ ]:
# revenue
banco_pred.groupby('grupo')['valor_total'].sum().sort_values(ascending=False)

In [ ]:
faturamento_por_grupo = banco_pred.groupby('grupo')['valor_total'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,6))
plt.bar(faturamento_por_grupo.index, faturamento_por_grupo.values)
plt.title('revenue by group')
plt.xlabel('group')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45)
plt.ylabel('revenue')
plt.show()

---

### Average Ticket by Group

In [ ]:
resumo_grupos = banco.groupby('grupo').agg(vendas=('grupo', 'size'), quantidade=('quantidade', 'sum'), 
                                           faturamento=('valor_total', 'sum')).sort_values('revenue', ascending=False)


resumo_grupos

ANTENA has a relatively balanced relationship between transaction count and units sold. COMPONENTES ANTENAS and CABO are complementary to antenna installations.

CABO is more difficult to forecast because it has many units sold but relatively low revenue, which is consistent with length-based sales. SEGURANCA has fewer transactions but high quantity and revenue, while INTERFONE, RECEPTOR, CONVERSOR, PORTAO, and SUPORTE TELEVISÃO are comparatively balanced groups for seasonality analysis.

In [ ]:
resumo_grupos['ticket_medio'] = resumo_grupos['revenue'] / resumo_grupos['sales']

resumo_grupos.sort_values('ticket_medio', ascending=False)

The highest average tickets are associated with RECEPTOR, INTERFONE, SEGURANCA, and PORTAO. Although ANTENA, COMPONENTES ANTENAS, and CABO account for many transactions, they are not necessarily the company's main revenue drivers.

---

### Item Revenue Analysis by Group

In [ ]:
banco_pred[banco_pred['grupo'] == 'SEGURANCA'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

SEGURANCA includes products sold in large quantities or by length, such as steel cable, electric-fence cable, and installation components. These products explain the group's high quantity and revenue.

In [ ]:
banco_pred[banco_pred['grupo'] == 'CABO'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

CABO products are frequently sold by length. This makes units-sold comparisons less direct, but the group may still provide useful context for ANTENA and COMPONENTES ANTENAS sales.

In [ ]:
banco_pred[banco_pred['grupo'] == 'COMPONENTES ANTENAS'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

In [ ]:
banco_pred[banco_pred['grupo'] == 'PORTAO'].groupby('descricao').agg(
        vendas=('descricao','size'),
        quantidade=('quantidade','sum'),
        faturamento=('valor_total','sum')
    ).sort_values('revenue', ascending=False)

---

### Revenue by Group and Year

In [ ]:
faturamento_grupo_ano = banco_pred.groupby([banco_pred['data_venda'].dt.year,'grupo'])['valor_total'].sum().reset_index()

faturamento_grupo_ano.columns = ['ano','grupo','revenue']

pivot = faturamento_grupo_ano.pivot(
    index='ano',
    columns='grupo',
    values='revenue'
)

pivot.plot(
    figsize=(12,6),
    marker='o'
)

plt.title('Revenue by group over the years')
plt.xlabel('year')
plt.ylabel('revenue')
plt.grid(True)
plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
plt.show()

---

### Correlation Between Sales Across Groups

In [ ]:
# Monthly revenue time series by group

faturamento_grupos = (
banco_pred
    .groupby([
        banco_pred['data_venda'].dt.to_period('M'), 'grupo'
    ])['valor_total'].sum().unstack(fill_value=0)
)
# unstack() cria "table dynamics" a partir of the result of the groupby, where the index is a sales records monthly 
# with groups as columns and monthly revenue as values.    
# Missing group-month combinations are filled with zero.

faturamento_grupos.index = (
    faturamento_grupos.index.to_timestamp()
)

faturamento_grupos.head()


In [ ]:
# correlation between sales monthly of the groups

corr = faturamento_grupos.corr()

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm'
)

---

## Temporal Analysis

#### Checking Record Consistency

In [ ]:
banco_pred['ano'] = banco_pred['data_venda'].dt.year
banco_pred['mes'] = banco_pred['data_venda'].dt.month

In [ ]:
# Count sales by year and month
vendas_mes_ano = (
    banco_pred.groupby(['ano', 'mes'])
    .size()
    .reset_index(name='qtd_vendas')
)
# reset_index converts the count series into a DataFrame, 
# where the new column contains the count values.
vendas_mes_ano.head()

In [ ]:
pivot = vendas_mes_ano.pivot(
    index='mes',
    columns='ano',
    values='qtd_vendas'
)

pivot.plot(figsize=(12,6), marker='o')

plt.title('Number of sales by month and year')
plt.xlabel('month')
plt.ylabel('number of sales')
plt.xticks(range(1,13))
plt.grid(True)
plt.show()

Monthly sales counts reveal an incomplete first year in 2017 and an abrupt decline after August 2025. The company reported that September and October 2025 were affected by a system migration, so those records are removed to prevent artificial noise in the time series.

The annual patterns also show the effects of the 2019 decline, the 2020 pandemic, the relative stability from 2021 to 2024, and the incomplete 2025 coverage.

In [ ]:
print(f'First recorded sale date: {banco_pred["data_venda"].min()}')
print(f'Last recorded sale date: {banco_pred["data_venda"].max()}')

In [ ]:
banco_pred.groupby(
    banco_pred['data_venda'].dt.to_period('M')
).size()

Only three sales are recorded in September 2025 and three in October 2025. These months are removed to avoid distorting the time series.

In [ ]:
# Removing the records of September and October of 2025
banco_pred = banco_pred[
    ~(
        (banco_pred['data_venda'].dt.year == 2025)
        &
        (banco_pred['data_venda'].dt.month.isin([9, 10]))
    )
].copy()

In [ ]:
# After removing the 2025 records
vendas_mes_ano_1 = (
    banco_pred.groupby(['ano', 'mes'])
    .size()
    .reset_index(name='qtd_vendas')
)

pivot_1 = vendas_mes_ano_1.pivot(
    index='mes',
    columns='ano',
    values='qtd_vendas'
)

pivot_1.plot(figsize=(12,6), marker='o')

plt.title('Number of sales by month and year')
plt.xlabel('month')
plt.ylabel('number of sales')
plt.xticks(range(1,13))
plt.grid(True)
plt.show()

In [ ]:
vendas_por_mes = (
    banco_pred.groupby(
        banco_pred['data_venda'].dt.month
    )
    .size()
)

plt.bar(vendas_por_mes.index, vendas_por_mes.values)
plt.title('Number of sales by month')
plt.xlabel('month')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(range(1,13))
plt.ylabel('number of sales')
plt.show()

Demand is lower at the beginning and end of the year, including January, February, September, October, November, and December. Sales are more balanced from March through August, which represents the strongest purchasing period in this market.

In [ ]:
vendas_por_ano = (
    banco_pred.groupby(
        banco_pred['data_venda'].dt.year
    )
    .size()
)

plt.bar(vendas_por_ano.index, vendas_por_ano.values)
plt.title('Number of sales by year')
plt.xlabel('year')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(vendas_por_ano.index)
plt.ylabel('number of sales')
plt.show()

2019 is the weakest complete year in the dataset, while 2025 is incomplete because of the system migration. These effects must be considered when comparing annual results.

### Revenue

In [ ]:
faturamento_ano = (
    banco_pred.groupby(
        banco_pred['data_venda'].dt.year
    )['valor_total']
    .sum()
)

faturamento_ano.plot(
    kind='bar',
    figsize=(10,5)
)
plt.ylabel('revenue')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.title('revenue by year')
plt.show()

Annual revenue complements the annual transaction count. The 2019 decline is visible in revenue, revenue increases after 2019, and the lower 2025 value is expected because four months of records are unavailable.

In [ ]:
faturamento_mes = (
    banco_pred.groupby(
        banco_pred['data_venda'].dt.month
    )['valor_total']
    .sum()
)

faturamento_mes.plot(
    kind='bar',
    figsize=(10,5)
)
plt.title('revenue by month')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylabel('revenue')
plt.show()

Monthly revenue complements the monthly sales count. Revenue is generally stronger in the middle of the year and lower at the beginning and end, which is consistent with the observed seasonal pattern.

#### Monthly Revenue Time Series

The monthly revenue series is evaluated for trend, seasonality, and noise.

In [ ]:
faturamento_mensal = (
    banco_pred.groupby(
        banco_pred['data_venda'].dt.to_period('M')
    )['valor_total']
    .sum()
)

faturamento_mensal.index = (
    faturamento_mensal.index.to_timestamp()
)

plt.figure(figsize=(12,6))
plt.plot(faturamento_mensal)
plt.grid(True)
plt.title('revenue monthly')
plt.ylabel('revenue')
plt.show()

* A visible upward trend is present in the revenue series.

In [ ]:
fat_mes_ano = (
    banco_pred
    .assign(
        ano=banco_pred['data_venda'].dt.year,
        mes=banco_pred['data_venda'].dt.month
    )
    .groupby(['ano','mes'])['valor_total']
    .sum()
    .reset_index() # .reset_index() converts the result index 
    # from groupby into regular columns
)

faturamento_medio_mes = (
    fat_mes_ano.groupby('mes')['valor_total']
    .mean()
)

plt.figure(figsize=(10,6))
plt.plot(faturamento_medio_mes.index, faturamento_medio_mes.values)
plt.title('Average monthly revenue')
plt.xlabel('month')
plt.xticks(range(1,13))
plt.ylabel('revenue')
plt.show()

In [ ]:
pivot_fat = fat_mes_ano.pivot(
    index='mes',
    columns='ano',
    values='valor_total'
)

plt.figure(figsize=(12,6))
sns.heatmap(
    pivot_fat,
    annot=True,
    fmt='.0f',
    cmap='YlGnBu'
)

plt.title('revenue by month and year')
plt.show()

The average revenue is highest in April and May. Revenue increases substantially after May 2021 and stabilizes from 2022 onward.

#### Is there seasonality after removing the trend?

In [ ]:
decomp_add = seasonal_decompose(
    faturamento_mensal,
    model='additive',
    period=12
)

decomp_add.plot();

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(
    4, 1,
    figsize=(14, 10),
    sharex=True
)

componentes = [
    faturamento_mensal,
    decomp_add.trend,
    decomp_add.seasonal,
    decomp_add.resid
]

titulos = [
    'revenue monthly',
    'trend',
    'seasonality',
    'residuals'
]


for ax, serie, titulo in zip(axes, componentes, titulos):
    ax.plot(
        serie.index,
        serie.values,
        linewidth=2
    )

    ax.set_title(
        titulo,
        fontsize=12,
        fontweight='bold'
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


axes[-1].xaxis.set_major_locator(
    mdates.YearLocator()
)

axes[-1].xaxis.set_major_formatter(
    mdates.DateFormatter('%Y')
)

plt.suptitle(
    'Additive decomposition of monthly revenue',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

In [ ]:
decomp_mul = seasonal_decompose(
    faturamento_mensal,
    model='multiplicative',
    period=12
)

decomp_mul.plot();

In [ ]:
import matplotlib.dates as mdates

fig, axes = plt.subplots(
    4, 1,
    figsize=(14, 10),
    sharex=True
)

componentes = [
    faturamento_mensal,
    decomp_mul.trend,
    decomp_mul.seasonal,
    decomp_mul.resid
]

titulos = [
    'revenue monthly',
    'trend',
    'seasonality',
    'residuals'
]


for ax, serie, titulo in zip(axes, componentes, titulos):
    ax.plot(
        serie.index,
        serie.values,
        linewidth=2
    )

    ax.set_title(
        titulo,
        fontsize=12,
        fontweight='bold'
    )

    ax.grid(
        True,
        alpha=0.3
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


axes[-1].xaxis.set_major_locator(
    mdates.YearLocator()
)

axes[-1].xaxis.set_major_formatter(
    mdates.DateFormatter('%Y')
)

plt.suptitle(
    'Multiplicative decomposition of monthly revenue',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

The seasonal index is calculated for each calendar month.

In [ ]:
plt.figure(figsize=(10,5))

plt.bar(
    indices_sazonais['mes'],
    indices_sazonais['indice_sazonal']
)

plt.axhline(
    y=1,
    color='red',
    linestyle='--'
)

plt.xlabel('month')
plt.ylabel('Seasonal index')
plt.title('Seasonal index by month')

plt.show()

In [ ]:
sns.boxplot(
    data=fat_mes_ano,
    x='mes',
    y='valor_total'
)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(faturamento_mensal, lags=36)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

resultado = adfuller(faturamento_mensal)

print('ADF:', resultado[0])
print('p-value:', resultado[1])

In [ ]:
serie_diff = faturamento_mensal.diff().dropna()

resultado = adfuller(serie_diff)

print('p-value after differencing:', resultado[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 1, figsize=(10,8))

plot_acf(serie_diff, lags=36, ax=ax[0])
plot_pacf(serie_diff, lags=36, ax=ax[1])

plt.show()

---

## Forecasting Algorithms

In [ ]:
faturamento_mensal = (
    banco_pred
    .groupby(
        banco_pred['data_venda'].dt.to_period('M')
    )['valor_total']
    .sum()
)

faturamento_mensal.index = (
    faturamento_mensal.index.to_timestamp()
)

In [ ]:
n = len(faturamento_mensal)

treino = faturamento_mensal[:int(0.8*n)]
teste = faturamento_mensal[int(0.8*n):]

### Baseline (Naive Forecast)

In [ ]:
y_train = treino
y_test = teste

y_pred_naive = y_test.shift(1)

# The first forecast uses the last training value
y_pred_naive.iloc[0] = y_train.iloc[-1]

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred_naive)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_naive))

print(mae)
print(rmse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,5))

plt.plot(y_train.index, y_train, label='training')
plt.plot(y_test.index, y_test, label='Actual')
plt.plot(y_pred_naive.index, y_pred_naive, label='Baseline')

plt.legend()
plt.show()

### Holt-Winters

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

modelo_hw = ExponentialSmoothing(
    treino,
    trend='add',
    seasonal='mul',
    seasonal_periods=12
)

ajuste_hw = modelo_hw.fit()

pred_hw = ajuste_hw.forecast(len(teste))

In [ ]:
mae_hw = mean_absolute_error(teste, pred_hw)
rmse_hw = np.sqrt(mean_squared_error(teste, pred_hw))

print(f'MAE: {mae_hw:.3f}')
print(f'RMSE: {rmse_hw:.3f}')

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(treino.index, treino, label='training')
plt.plot(teste.index, teste, label='test')
plt.plot(pred_hw.index, pred_hw, label='Holt-Winters')

plt.legend()
plt.show()

### ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

modelo = ARIMA(
    treino,
    order=(0,1,1)
)

ajuste = modelo.fit()

pred_arima = ajuste.forecast(
    steps=len(teste)
)

In [ ]:
mae_arima = mean_absolute_error(teste, pred_arima)
rmse_arima = np.sqrt(mean_squared_error(teste, pred_arima))

print(f'MAE: {mae_arima:.3f}')
print(f'RMSE: {rmse_arima:.3f}')

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(treino.index, treino, label='training')
plt.plot(teste.index, teste, label='test')
plt.plot(pred_arima.index, pred_arima, label='ARIMA')

plt.legend()
plt.show()

### Auto-ARIMA

In [ ]:
from pmdarima import auto_arima

modelo_arima = auto_arima(
    treino,
    seasonal=True,
    m=12,
    stepwise=True,
    trace=True,
    suppress_warnings=True
)

print(modelo_arima.summary())

In [ ]:
pred_arima = modelo_arima.predict(
    n_periods=len(teste)
)

In [ ]:
pred_arima = pd.Series(
    pred_arima,
    index=teste.index
)

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

import numpy as np

mae = mean_absolute_error(
    teste,
    pred_arima
)

rmse = np.sqrt(
    mean_squared_error(
        teste,
        pred_arima
    )
)

mape = (
    np.abs(
        (teste - pred_arima)
        / teste
    ).mean()
) * 100

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))

plt.plot(
    treino.index,
    treino,
    label='training'
)

plt.plot(
    teste.index,
    teste,
    label='Actual'
)

plt.plot(
    pred_arima.index,
    pred_arima,
    label='ARIMA'
)

plt.title('ARIMA - forecast vs Actual')
plt.ylabel('revenue')
plt.xlabel('data')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    teste.index,
    teste,
    marker='o',
    label='Actual'
)

plt.plot(
    pred_arima.index,
    pred_arima,
    marker='s',
    label='Previsto'
)

plt.title('ARIMA - period of test')

plt.legend()
plt.grid(True)

plt.show()